## This lecture is about LANGUAGE MODELS

**Previously:** Text processing (L2) · Parsing (L3) · Word representation (L4) · Intro to ML (L5)

**Outline:**
- What modeling and language modeling are; are LMs "conscious"?
- **Statistical LMs:** n-grams, counting, backoff, linear interpolation, smoothing, generation, shortcomings
- **Neural LMs:** LM as classification, feed-forward NN, RNN, CNN (kernel, stride, padding, pooling, residuals), Transformers
- **Transformers as LMs:** encoder-decoder (**T5**), encoder-only (**BERT** family), decoder-only (**GPT**); logits
- **Evaluation:** extrinsic vs intrinsic; **cross entropy**, **perplexity**, **BLEU**
- **Implementation strategies:** training from scratch, fine-tuning (full, partial, feature-based, adapters, heads), feature extraction, prompt engineering

**Announcements:** Assignment 5 (the **last** one) is due **Nov 16, 11:59pm ET**. Keep submitting the weekly **final-project progress update** (one-pager). Remaining schedule: NER, Q&A, Machine Translation, LLMs → final project presentations (two sessions) → full submission deadline.

In [1]:
import re, math
from collections import Counter, defaultdict
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

# Modeling and Language Modeling
**Modeling:** the process of creating an (abstract) representation of a complex phenomenon, with the aim of understanding/analyzing or predicting/controlling its behavior.
e.g. an **electrical circuit model of the heart's left ventricle**: resistors and capacitors stand in for vessels and tissue.

**Language modeling (LM):** model a **linguistic unit** (text, sentence, token, symbol).
- A specific area of NLP and AI whose goal is to develop a computational model that can **understand, interpret, generate, and respond to** human language.
- Creates a mathematical representation of a language's **syntax, vocabulary, grammar, and semantics**.

### Addressing a myth: are LMs conscious? Do they think like humans?
**No.** They are not conscious and do not "think" like humans, but they can perform forms of **reasoning** by learning the statistical structure of language and knowledge.

- LLMs (GPT-4, LLaMA, …) are trained with a very simple objective, **predict the next token**:
$$P(w_t \mid w_1, w_2, \dots, w_{t-1})$$
- They see billions or trillions of examples and adjust millions or billions of parameters so their probabilities match the patterns in the data.
- In a strict sense they learn **statistical distributions of language**; they have no goals, consciousness, or understanding.

**Why do they still appear to "think"?** Language itself encodes facts, logic, cause–effect relationships, mathematical reasoning, and structured arguments. Learning the statistical structure of language at scale **indirectly** learns the structure of reasoning embedded in it.
> All mammals breathe. A whale is a mammal. Therefore a whale breathes.

The model can reproduce this kind of reasoning without ever being explicitly programmed with logic rules.
- Humans: **experience → abstraction → reasoning**
- LLMs: **data → statistical structure → reasoning behavior**

# Types of language models
- **Statistical LMs**: earlier models based on statistics like **n-grams**, where each word's probability depends on the preceding $n-1$ words. Simple, but limited in handling long-range dependencies and contextual nuance.
- **Neural LMs**: deep learning to capture complex patterns: RNNs, LSTMs and, more recently, Transformers like **BERT** and **GPT**.

# Statistical Language Models
**Goal:** estimate the probabilities of text fragments (word, sentence, paragraph, …).

By the **chain rule** of probability, the probability of a sentence $I = (y_1, \dots, y_n)$ is
$$P(y_1, y_2, \dots, y_n) = P(y_1)\cdot P(y_2 \mid y_1)\cdot P(y_3 \mid y_1, y_2)\cdots P(y_n \mid y_1, \dots, y_{n-1}) = \prod_{t=1}^{n} P(y_t \mid y_{<t})$$

## n-grams
- An **n-gram** is a contiguous sequence of $n$ items from a text: unigram ($n=1$), bigram ($n=2$), trigram ($n=3$).
- **Word n-grams:** for "The quick brown fox" the bigrams are "The quick", "quick brown", "brown fox".
- **Character n-grams:** the trigrams of "chat" are "cha" and "hat".

## Estimating the probabilities: count!
Estimate from **global statistics** of a text corpus:
$$P(y_t \mid y_1, \dots, y_{t-1}) = \frac{N(y_1, \dots, y_{t-1}, y_t)}{N(y_1, \dots, y_{t-1})}$$
where $N(y_1, \dots, y_k)$ is the number of times the sequence $(y_1, \dots, y_k)$ occurs in the text.

Long histories almost never repeat, so the **Markov assumption** is used: only the last $n-1$ tokens matter.
- **Unigram** ($n=1$): $P(y_t \mid y_1, \dots, y_{t-1}) = P(y_t)$
- **Bigram** ($n=2$): $P(y_t \mid y_1, \dots, y_{t-1}) = P(y_t \mid y_{t-1})$
- **Trigram** ($n=3$): $P(y_t \mid y_1, \dots, y_{t-1}) = P(y_t \mid y_{t-2}, y_{t-1})$

$$P(y_t \mid y_1, \dots, y_{t-1}) = \frac{N(y_{t-n+1}, \dots, y_{t-1}, y_t)}{N(y_{t-n+1}, \dots, y_{t-1})}$$

In [2]:
def ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

print('word bigrams :', ngrams("The quick brown fox".split(), 2))
print('char trigrams:', [''.join(g) for g in ngrams(list("chat"), 3)])

word bigrams : [('The', 'quick'), ('quick', 'brown'), ('brown', 'fox')]
char trigrams: ['cha', 'hat']


In [3]:
corpus = """the cat sat on a mat .
the dog sat on a rug .
a cat is on a mat .
the cat lay on the mat .
a dog sat on the mat .
the cat ate the fish .
a dog chased the cat .
the dog is on a rug ."""

sents = [['<s>', '<s>'] + s.split() + ['</s>'] for s in corpus.split('\n')]
counts = {n: Counter(g for s in sents for g in ngrams(s, n)) for n in (1, 2, 3, 4)}
V = sorted({w for s in sents for w in s if w != '<s>'})
print('vocabulary size |V| =', len(V))

# denominator = how often the context is followed by a vocabulary word  (= N(context) in the formula)
context_total = defaultdict(int)
for n, cnt in counts.items():
    for g, c in cnt.items():
        if g[-1] != '<s>':
            context_total[g[:-1]] += c

def p_mle(word, context):
    """P(word | context) = N(context + word) / N(context)"""
    context = tuple(context)
    num = counts[len(context) + 1][context + (word,)]
    den = context_total[context]
    return num / den if den else None          # None = context never seen (can't compute)

print('P(mat | on a)     =', p_mle('mat', ['on', 'a']))
print('P(rug | on a)     =', p_mle('rug', ['on', 'a']))
print('P(mat | a)        =', p_mle('mat', ['a']))
print('P(mat)            =', round(p_mle('mat', []), 4))
print('P(mat | cat on a) =', p_mle('mat', ['cat', 'on', 'a']), ' <- N(cat on a) = 0 in this corpus!')

vocabulary size |V| = 15
P(mat | on a)     = 0.5
P(rug | on a)     = 0.5
P(mat | a)        = 0.2857142857142857
P(mat)            = 0.0645
P(mat | cat on a) = None  <- N(cat on a) = 0 in this corpus!


## What if $N(y_{t-n+1}, \dots, y_{t-1}) = 0$? (the **context** was never seen)
### Backoff: use a smaller window
$P(\text{mat} \mid \text{cat on a}) = \frac{N(\text{cat on a mat})}{N(\text{cat on a})} = \frac{?}{0}$ → can't compute. So:
- $N(\text{cat on a}) = 0$ → try "on a": $P(\text{mat} \mid \text{cat on a}) \approx P(\text{mat} \mid \text{on a})$
- $N(\text{on a}) = 0$ → try "a": $P(\text{mat} \mid \text{on a}) \approx P(\text{mat} \mid \text{a})$
- $N(\text{a}) = 0$ → try the unigram: $P(\text{mat} \mid \text{a}) \approx P(\text{mat})$

### Linear interpolation: mix all the probabilities
$$\hat{P}(\text{mat} \mid \text{cat on a}) \approx \lambda_3 P(\text{mat} \mid \text{cat on a}) + \lambda_2 P(\text{mat} \mid \text{on a}) + \lambda_1 P(\text{mat} \mid \text{a}) + \lambda_0 P(\text{mat}), \qquad \sum_i \lambda_i = 1$$
The coefficients $\lambda_i$ can be picked by **cross-validation** on the development set.

## What if $N(y_{t-n+1}, \dots, y_{t-1}, y_t) = 0$? (the **n-gram** was never seen → probability 0)
### Smoothing: just add 1 to all counts (Laplace), or a small $\delta$
$$\hat{P}(\text{mat} \mid \text{cat on a}) = \frac{\delta + N(\text{cat on a mat})}{\delta \cdot |V| + N(\text{cat on a})}$$

In [4]:
def p_backoff(word, context):
    context = list(context)
    while True:
        p = p_mle(word, context)
        if p is not None:
            return p, ' '.join(context) or '(unigram)'
        context = context[1:]                       # drop the oldest word

def p_interp(word, context, lambdas=(0.1, 0.2, 0.3, 0.4)):   # λ0 (unigram) ... λ3 (longest)
    assert abs(sum(lambdas) - 1) < 1e-9
    total = 0
    for k, lam in enumerate(lambdas):               # k = context length used
        if k > len(context): continue
        ctx = list(context)[len(context) - k:] if k else []
        total += lam * (p_mle(word, ctx) or 0)      # unseen context contributes 0
    return total

def p_smooth(word, context, delta=1):
    context = tuple(context)
    return (delta + counts[len(context) + 1][context + (word,)]) / (delta * len(V) + context_total[context])

ctx = ['cat', 'on', 'a']
p, used = p_backoff('mat', ctx)
print(f'backoff      P(mat | cat on a) ≈ P(mat | {used}) = {p:.4f}')
print(f'interpolated P(mat | cat on a) = {p_interp("mat", ctx):.4f}')
print()
print('unseen n-gram: N(on a fish) = 0')
print(f'  MLE                 P(fish | on a) = {p_mle("fish", ["on", "a"])}')
print(f'  add-1 smoothing     P(fish | on a) = {p_smooth("fish", ["on", "a"], 1):.4f}')
print(f'  add-0.1 smoothing   P(fish | on a) = {p_smooth("fish", ["on", "a"], 0.1):.4f}')
print(f'  (seen) add-1        P(mat  | on a) = {p_smooth("mat", ["on", "a"], 1):.4f}  vs MLE {p_mle("mat", ["on", "a"]):.4f}')
print('sum over vocab after add-1:', round(sum(p_smooth(w, ['on', 'a'], 1) for w in V), 6))

backoff      P(mat | cat on a) ≈ P(mat | on a) = 0.5000
interpolated P(mat | cat on a) = 0.2136

unseen n-gram: N(on a fish) = 0
  MLE                 P(fish | on a) = 0.0
  add-1 smoothing     P(fish | on a) = 0.0526
  add-0.1 smoothing   P(fish | on a) = 0.0182
  (seen) add-1        P(mat  | on a) = 0.1579  vs MLE 0.5000
sum over vocab after add-1: 1.0


## N-gram language generation
Start with a prompt (e.g. "I ___"), then repeatedly **sample** the next word from $P(y_t \mid \text{previous } n-1 \text{ words})$, append it, and repeat until the end-of-sentence token.

**Examples** (a 3-gram model trained on 2.5 million English sentences, from Lena Voita's course):
> *so even when i talk a bit short , there was no easy thing to do different buffer flushing strategies in the future , due to huge list of number - one just has started production of frits in the process and has free wi - fi " operation .... _eos_*
>
> *john holmes is a crystal - clear spring of 2001 . _eos_*

Locally plausible, but **NOT FLUENT** over longer spans.

## Shortcomings of n-gram LMs
- **Storage:** must store counts for all possible n-grams; model size $O(\exp(n))$.
- **Limited window size** limits context → poor **fluency**.

In [5]:
def generate(prefix=('the',), max_len=15, seed=0):
    rng = np.random.default_rng(seed)
    out = ['<s>', '<s>'] + list(prefix)
    while len(out) < max_len + 2:
        ctx = tuple(out[-2:])                              # trigram model: last 2 words
        cands = {g[-1]: c for g, c in counts[3].items() if g[:2] == ctx}
        if not cands:
            break
        words, freqs = zip(*cands.items())
        nxt = rng.choice(words, p=np.array(freqs) / sum(freqs))
        if nxt == '</s>':
            break
        out.append(nxt)
    return ' '.join(out[2:])

for seed in range(5):
    print(generate(('the',), seed=seed))

print('\nstored n-gram counts:', {n: len(c) for n, c in counts.items()})
print(f'possible trigrams over |V| = {len(V)}: {len(V) ** 3:,}; for a 50k vocab: {50_000 ** 3:,.0e}')

the dog sat on a rug .
the cat .
the cat lay on the mat .
the cat sat on a mat .
the dog is on a rug .

stored n-gram counts: {1: 16, 2: 30, 3: 39, 4: 44}
possible trigrams over |V| = 15: 3,375; for a 50k vocab: 1e+14


# Neural Language Models
**How do we compute the conditional probabilities?** Rather than *calculating* them from counts, **predict** them:
- i.e. **train an ML/DL model**
- i.e. a **classification problem**: the classes are the $|V|$ words of the vocabulary.

### The classifier recipe
- **Step 0:** initialize with input **embeddings**.
- **Step 1:** feed the embeddings of the previous (context) words into a network.
- **Step 2:** the network creates a **vector representation of the context**.
- **Step 3:** predict a **probability distribution** for the next token: a linear layer to $|V|$ scores (**logits**) + **softmax**.

## Feed-forward NN (fixed window; Bengio et al., 2003)
Context words $x^{<1>}, x^{<2>}, x^{<3>}$ → **projection layer** (look up and concatenate embeddings) → **hidden layer** $h$ ($d_h \times 1$) → **output layer** $y$ ($|V| \times 1$, softmax). Related to **word2vec** (L4), which uses the same "predict a word from its neighbors" idea.

**Limitations:**
- the **fixed window** ignores more distant context
- learning a **fixed representation** per position ignores how context changes meaning

In [6]:
def softmax(z):
    z = z - z.max(-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(-1, keepdims=True)

# Fixed-window feed-forward neural LM, trained from scratch with NumPy (window = 2 previous words)
words = sorted({w for s in sents for w in s})
w2i = {w: i for i, w in enumerate(words)}
Vn, d, dh, window = len(words), 12, 32, 2
train = [([w2i[w] for w in s[i - window:i]], w2i[s[i]]) for s in sents for i in range(window, len(s))]
Xc = np.array([c for c, _ in train]); Yt = np.array([t for _, t in train])

rng = np.random.default_rng(0)
E  = rng.normal(0, 0.1, (Vn, d))               # step 0: embeddings
W1 = rng.normal(0, 0.1, (window * d, dh)); b1 = np.zeros(dh)
W2 = rng.normal(0, 0.1, (dh, Vn));         b2 = np.zeros(Vn)

lr = 0.5
for epoch in range(400):
    x = E[Xc].reshape(len(Xc), -1)              # step 1: projection layer (concatenate embeddings)
    h = np.tanh(x @ W1 + b1)                    # step 2: context representation
    P = softmax(h @ W2 + b2)                    # step 3: distribution over the vocabulary
    loss = -np.log(P[np.arange(len(Yt)), Yt]).mean()
    # backprop (cross-entropy + softmax gradient = P - onehot)
    g = P.copy(); g[np.arange(len(Yt)), Yt] -= 1; g /= len(Yt)
    gW2, gb2 = h.T @ g, g.sum(0)
    gh = g @ W2.T * (1 - h ** 2)
    gW1, gb1 = x.T @ gh, gh.sum(0)
    gx = (gh @ W1.T).reshape(len(Xc), window, d)
    gE = np.zeros_like(E); np.add.at(gE, Xc, gx)
    for p_, g_ in [(W2, gW2), (b2, gb2), (W1, gW1), (b1, gb1), (E, gE)]:
        p_ -= lr * g_
    if epoch % 100 == 0 or epoch == 399:
        print(f'epoch {epoch:3d}  cross-entropy {loss:.3f}  perplexity {np.exp(loss):.2f}')

def ffnn_next(context):
    x = E[[w2i[w] for w in context]].reshape(1, -1)
    p = softmax(np.tanh(x @ W1 + b1) @ W2 + b2)[0]
    top = np.argsort(-p)[:3]
    return [(words[i], round(float(p[i]), 2)) for i in top]

for ctx in [['on', 'a'], ['the', 'cat'], ['cat', 'on']]:
    print(f'P(next | {" ".join(ctx):7}) top-3:', ffnn_next(ctx))

epoch   0  cross-entropy 2.775  perplexity 16.03
epoch 100  cross-entropy 0.590  perplexity 1.80
epoch 200  cross-entropy 0.448  perplexity 1.56
epoch 300  cross-entropy 0.408  perplexity 1.50
epoch 399  cross-entropy 0.395  perplexity 1.48
P(next | on a   ) top-3: [('mat', 0.5), ('rug', 0.49), ('on', 0.0)]
P(next | the cat) top-3: [('.', 0.26), ('lay', 0.24), ('sat', 0.24)]
P(next | cat on ) top-3: [('the', 1.0), ('a', 0.0), ('on', 0.0)]


## RNN-based LM
At each time step the RNN combines the previous hidden state with the current input:
$$a^{<t>} = g_1\big(W_a \cdot [a^{<t-1>}, x^{<t>}] + b_a\big), \qquad \hat{y}^{<t>} = g_2\big(W_y \cdot a^{<t>} + b_y\big)$$
($g_1$ usually tanh, $g_2$ softmax over the vocabulary.) It reads $x^{<1>}, x^{<2>}, \dots, x^{<k-1>}$ and at every step predicts the next token. **Multi-layer** RNNs stack several recurrent layers.

| ✅ Advantages | ❌ Disadvantages |
|---|---|
| can process **any length** input | **sequential** processing → slow (can't parallelize over time) |
| model size doesn't grow with input length | **limited context** in practice (vanishing gradients) |
| step $t$ can (in theory) use information from many steps back | |
| weights are **shared** across time steps → shared representations | |

In [7]:
# One RNN, the same weights reused at every time step (forward pass with random weights)
rng = np.random.default_rng(1)
da = 8
Wa = rng.normal(0, 0.3, (da, da + d)); ba = np.zeros(da)
Wy = rng.normal(0, 0.3, (Vn, da));     by = np.zeros(Vn)

sentence = ['<s>', 'the', 'cat', 'sat', 'on', 'a']
a = np.zeros(da)
for t, w in enumerate(sentence, 1):
    a = np.tanh(Wa @ np.concatenate([a, E[w2i[w]]]) + ba)     # a<t> = g1(Wa·[a<t-1>, x<t>] + ba)
    y = softmax(Wy @ a + by)                                   # y<t> = g2(Wy·a<t> + by)
    print(f't={t} input={w:4}  |a|={np.linalg.norm(a):.2f}  P(next) sums to {y.sum():.1f}')
print('number of parameters:', Wa.size + ba.size + Wy.size + by.size, '(the same for any sentence length)')

t=1 input=<s>   |a|=1.66  P(next) sums to 1.0
t=2 input=the   |a|=1.50  P(next) sums to 1.0
t=3 input=cat   |a|=1.70  P(next) sums to 1.0
t=4 input=sat   |a|=1.43  P(next) sums to 1.0
t=5 input=on    |a|=1.85  P(next) sums to 1.0
t=6 input=a     |a|=1.51  P(next) sums to 1.0
number of parameters: 312 (the same for any sentence length)


## CNN-based LM
### Convolution for text
Apply a small **pattern detector** (filter) that slides across a sequence of word embeddings to detect meaningful **local patterns** (phrases).

**A typical model: convolution + pooling.** Each **filter** takes the vectors in the current window and transforms them linearly into a **single feature**.

| Term | Meaning |
|---|---|
| **Kernel size** | number of input tokens the convolution looks at each step |
| **Stride** | how far the filter moves at each step |
| **Padding** | add zero vectors to both sides |
| **Multiple features** | several filters → several features per position, computed **in parallel** |
| **Multiple kernels** | filters with **different sizes** (e.g. 2, 3, 4 words) |

**Pooling:** summarizes the features in a region → the **strongest occurrence** of that pattern in the sentence; reduces dimension (fewer parameters). Types: **mean** and **max** pooling.

**Residual connections:** deep layers create a longer context window, but are hard to train. **Idea:** add the input of a block to its output, $y = x + F(x)$.

### CNN language model: putting it all together
- **Shift tokens to the right with padding** (pad only on the left = *causal* convolution), so position $t$ never sees future tokens.
- Do **NOT** remove positional information: **no pooling** (we need a prediction at every position).
- If you stack many layers, don't forget **residual connections**.

In [8]:
def conv1d(X, filters, kernel=3, stride=1, pad_left=0, pad_right=0):
    """X: (T, d) embeddings; filters: (n_filters, kernel*d). Returns (T_out, n_filters)."""
    Xp = np.vstack([np.zeros((pad_left, X.shape[1])), X, np.zeros((pad_right, X.shape[1]))])
    windows = [Xp[i:i + kernel].ravel() for i in range(0, len(Xp) - kernel + 1, stride)]
    return np.maximum(0, np.array(windows) @ filters.T)          # linear map + ReLU

rng = np.random.default_rng(2)
tokens = ['the', 'cat', 'sat', 'on', 'the', 'mat']
X = E[[w2i[w] for w in tokens]]                                  # (6, d)
filters = rng.normal(0, 0.5, (4, 3 * d))                         # 4 features, kernel size 3

print('no padding, stride 1      ->', conv1d(X, filters).shape)
print('no padding, stride 2      ->', conv1d(X, filters, stride=2).shape)
print('"same" padding (1 + 1)    ->', conv1d(X, filters, pad_left=1, pad_right=1).shape)
print('causal padding (2 left)   ->', conv1d(X, filters, pad_left=2).shape, ' <- one output per token, sees only the past')

F = conv1d(X, filters, pad_left=1, pad_right=1)
print('\nmax pooling over time (classification):', F.max(0).round(2))
print('mean pooling over time                 :', F.mean(0).round(2))

# residual block: output = input + conv(input)  (filters map back to d dims)
f_res = rng.normal(0, 0.1, (d, 3 * d))
Y = X + conv1d(X, f_res, pad_left=2)
print('\nresidual block output shape:', Y.shape, '(same as input, so blocks can be stacked)')

no padding, stride 1      -> (4, 4)
no padding, stride 2      -> (2, 4)
"same" padding (1 + 1)    -> (6, 4)
causal padding (2 left)   -> (6, 4)  <- one output per token, sees only the past

max pooling over time (classification): [0.26 1.03 1.48 1.71]
mean pooling over time                 : [0.04 0.24 0.5  0.65]

residual block output shape: (6, 12) (same as input, so blocks can be stacked)


# Transformers as Language Models
**Refresher:** an **encoder** (self-attention + feed-forward, sees the whole input) and a **decoder** (masked self-attention + encoder-decoder attention + feed-forward, generates left to right).

| Architecture | Family | Attention | Typical use |
|---|---|---|---|
| **Encoder + Decoder** | **T5**, BART | encoder: bidirectional; decoder: causal | text-to-text: translation, summarization |
| **Encoder only** | **BERT** family | **bidirectional** (sees left and right) | understanding: classification, NER, QA |
| **Decoder only** | **GPT** family | **causal / unidirectional** (sees only the left) | generation |

In [9]:
T = 5
bidirectional = np.ones((T, T), dtype=int)               # BERT: every token attends to every token
causal = np.tril(np.ones((T, T), dtype=int))             # GPT: token t attends to tokens <= t
print('BERT (encoder) attention mask:\n', bidirectional)
print('GPT (decoder) causal attention mask:\n', causal)

# masked self-attention scores -> softmax rows
scores = np.random.default_rng(0).normal(size=(T, T))
masked = np.where(causal == 1, scores, -np.inf)
print('causal attention weights (row t = where token t looks):\n', softmax(masked).round(2))

BERT (encoder) attention mask:
 [[1 1 1 1 1]
 [1 1 1 1 1]
 [1 1 1 1 1]
 [1 1 1 1 1]
 [1 1 1 1 1]]
GPT (decoder) causal attention mask:
 [[1 0 0 0 0]
 [1 1 0 0 0]
 [1 1 1 0 0]
 [1 1 1 1 0]
 [1 1 1 1 1]]
causal attention weights (row t = where token t looks):
 [[1.   0.   0.   0.   0.  ]
 [0.28 0.72 0.   0.   0.  ]
 [0.32 0.62 0.06 0.   0.  ]
 [0.15 0.18 0.22 0.46 0.  ]
 [0.1  0.43 0.06 0.15 0.27]]


## T5: Text-To-Text Transfer Transformer
- *"Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer"*, **Google, 2019**.
- Converts **all NLP tasks** into one **text-to-text** format: input and output are always strings.
- **Architecture:** encoder-decoder, **12 pairs** of encoder-decoder blocks; each block has self-attention, a feed-forward network, and (in the decoder) encoder-decoder attention.
- **Input/output:** add a **task-specific prefix** to the input, e.g. `"summarize: …"`, `"translate English to French: …"`.
- **Training data:** **C4** (Colossal Clean Crawled Corpus), **750 GB** of clean English web text.
- **Training objectives:** language modeling (predict the next word); **masking / replacing** words with random words and predicting the original text; **de-shuffling** (shuffle the input and predict the original).

In [ ]:
# !pip install transformers sentencepiece torch
from transformers import pipeline
t5 = pipeline("text2text-generation", model="t5-small")
print(t5("translate English to French: The patient has a fever."))
print(t5("summarize: The patient was admitted with chest pain. A CT scan showed no abnormality. "
         "She was discharged the next day with instructions to follow up with cardiology."))

## BERT: Bidirectional Encoder Representations from Transformers
- Introduced **October 2018** by Google: *"BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding"*.
- **Encoder-only**: a stack of Transformer encoders; **bidirectional**, so it sees the whole sentence (left and right context).
- Trained on the entire **Wikipedia** and **BookCorpus**.

### BERT training
1. **Masked word prediction (MLM):** hide some input tokens with `[MASK]` and predict them from both sides.
2. **Next sentence prediction (NSP):** predict whether sentence B really follows sentence A; **50%** correct pairs mixed with **50%** random pairs. Teaches relationships between sentences.

**Special symbols:**
- `[CLS]`: at the beginning of the input; its output vector represents the whole input for **classification** (id **101**).
- `[SEP]`: at the end of the input and **between sentences** (id **102**).

**Input** = token embeddings + segment embeddings (sentence A or B) + position embeddings. **Output** = a contextual vector for every token.

### BERT's family
- **RoBERTa** (Robustly Optimized BERT Pretraining Approach): improved training strategy; larger dataset, trained longer → better accuracy and robustness; excels at QA and natural language inference.
- **XLNet** (Generalized Autoregressive Pretraining): combines BERT's strengths with autoregressive language modeling; captures longer-range dependencies; strong at understanding and generation.

### BERT's use cases
**Classification** (from the `[CLS]` vector) · **Question answering** (predict the answer span's start/end) · **Named entity recognition** (a label for every token)

In [ ]:
from transformers import AutoTokenizer, pipeline
tok = AutoTokenizer.from_pretrained("bert-base-uncased")

enc = tok("The patient has a fever.", "She was given aspirin.")
print(tok.convert_ids_to_tokens(enc["input_ids"]))
print(enc["input_ids"])          # [CLS] = 101, [SEP] = 102
print(enc["token_type_ids"])     # segment A = 0, segment B = 1

fill = pipeline("fill-mask", model="bert-base-uncased")
for r in fill("The patient was given [MASK] for the pain.")[:5]:
    print(f"{r['token_str']:12} {r['score']:.3f}")

## GPT: Generative Pretrained Transformer
- Developed by **OpenAI, 2018**: *"Improving Language Understanding by Generative Pre-Training"*.
- **Decoder-only** Transformer; **unidirectional** training: predict each next word from the preceding ones.

**Why decoder-only?** A deliberate choice aligned with generating text **autoregressively**. It excels at a wide range of generation tasks without the computational overhead and complexity of an encoder-decoder.

**GPT-3 training facts (from the slides):**
- next-word prediction (**unsupervised / self-supervised**) on **300 billion tokens**
- vocabulary of **50,257** tokens (byte-level BPE, see L2)
- input sequence fixed to **2048** tokens; output: 2048 guesses (a probability distribution at every position)

**GPT-2 walkthrough** ([The Illustrated GPT-2](https://jalammar.github.io/illustrated-gpt2/)):
1. **Input embedding** = token embedding + positional encoding
2. pass up through the stack of **decoder blocks** (masked self-attention + feed-forward)
3. the top vector is multiplied by the embedding matrix → a score for every word in the vocabulary: the **logits**
4. softmax → probabilities → pick/sample the next token, append it, and repeat

### Logit
- The **raw output of the last linear layer** of a classification network, **before** the softmax.
- They are **unnormalized log probabilities**; the softmax converts them into actual probabilities.

In [10]:
logits = np.array([2.0, 1.0, 0.1, -1.0])
vocab4 = ['mat', 'rug', 'fish', 'cat']
probs = softmax(logits)
print('logits      :', logits)
print('softmax     :', probs.round(3), ' sum =', probs.sum())
print('log softmax :', np.log(probs).round(3), ' (= logits - logsumexp: same differences as the logits)')

# decoding strategies on top of the probabilities
rng = np.random.default_rng(0)
print('\ngreedy           :', vocab4[int(np.argmax(probs))])
for temp in [0.5, 1.0, 2.0]:
    p = softmax(logits / temp)
    print(f'temperature {temp:3}  : probs {p.round(2)}  sample -> {rng.choice(vocab4, p=p)}')

logits      : [ 2.   1.   0.1 -1. ]
softmax     : [0.638 0.235 0.095 0.032]  sum = 1.0
log softmax : [-0.449 -1.449 -2.349 -3.449]  (= logits - logsumexp: same differences as the logits)

greedy           : mat
temperature 0.5  : probs [0.86 0.12 0.02 0.  ]  sample -> mat
temperature 1.0  : probs [0.64 0.23 0.1  0.03]  sample -> mat
temperature 2.0  : probs [0.45 0.27 0.17 0.1 ]  sample -> mat


In [ ]:
# GPT-2 next-token logits and generation with Hugging Face
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained("gpt2")
gpt2 = AutoModelForCausalLM.from_pretrained("gpt2")

ids = tok("The patient was diagnosed with", return_tensors="pt").input_ids
with torch.no_grad():
    logits = gpt2(ids).logits[0, -1]                   # logits for the next token (size 50257)
top = torch.topk(torch.softmax(logits, -1), 5)
print([(tok.decode(i), round(p.item(), 3)) for p, i in zip(top.values, top.indices)])

out = gpt2.generate(ids, max_new_tokens=20, do_sample=True, top_k=50, pad_token_id=tok.eos_token_id)
print(tok.decode(out[0]))

# Language Model Evaluation
| | Extrinsic | Intrinsic |
|---|---|---|
| How | evaluate the LM on a **downstream task** | define a metric, test on a **held-out test corpus** |
| Examples | machine translation, sentence similarity, … | **cross entropy**, **perplexity**, **BLEU** |

## Cross entropy
**Definition:** a measure of the difference between two probability distributions, the **actual** one $p$ and the **predicted** one $q$:
$$H(p, q) = -\frac{1}{N}\sum_{i=1}^{N}\sum_{j=1}^{V} p(w_j \mid w_{1:i-1})\,\log q(w_j \mid w_{1:i-1})$$
- $p(w_j \mid w_{1:i-1})$: true distribution of the next word given the context (a one-hot vector in practice)
- $q(w_j \mid w_{1:i-1})$: distribution predicted by the model
- $N$: number of words in the evaluation set · $V$: vocabulary size

**Interpretation:** **lower** = predictions closer to the actual distribution (better next-word prediction); **higher** = less accurate.

In [11]:
def cross_entropy(P, Q):
    return -np.sum(P * np.log(Q))

# probabilities for true vs predicted values (slide example)
P = np.array([0, 1, 0, 0, 0])                    # True distribution
Q = np.array([0.1, 0.6, 0.1, 0.15, 0.05])        # Predicted distribution
print(f"Cross Entropy: {cross_entropy(P, Q):.2f}   (= -ln 0.6)")

for q_true in [0.99, 0.6, 0.2, 0.01]:
    print(f'  model gives the correct word p = {q_true:4}: cross entropy = {-np.log(q_true):.2f}')

Cross Entropy: 0.51   (= -ln 0.6)
  model gives the correct word p = 0.99: cross entropy = 0.01
  model gives the correct word p =  0.6: cross entropy = 0.51
  model gives the correct word p =  0.2: cross entropy = 1.61
  model gives the correct word p = 0.01: cross entropy = 4.61


## Perplexity
A way to capture the degree of **uncertainty** a model has when assigning probabilities to text.

### Worked example
**Model:** next-word prediction over a language of six words: "a", "the", "red", "fox", "dog", ".". **Test sentence:** "a red fox."

$$P(\text{"a red fox."}) = P(a)\cdot P(\text{red} \mid a)\cdot P(\text{fox} \mid \text{a red})\cdot P(. \mid \text{a red fox})$$

| | 'a' | 'the' | 'red' | 'fox' | 'dog' | '.' | picked |
|---|---|---|---|---|---|---|---|
| $P(w_1)$ | **0.4** | 0.3 | 0.15 | 0.08 | 0.07 | 0 | P('a') = 0.4 |
| $P(w_2 \mid a)$ | 0.01 | 0.01 | **0.27** | 0.3 | 0.4 | 0.01 | P('red' \| 'a') = 0.27 |
| $P(w_3 \mid \text{a red})$ | 0.02 | 0.03 | 0.03 | **0.55** | 0.22 | 0.15 | P('fox' \| 'a red') = 0.55 |
| $P(w_4 \mid \text{a red fox})$ | 0.02 | 0.03 | 0.03 | 0.02 | 0.11 | **0.79** | P('.' \| 'a red fox') = 0.79 |

$$P(\text{"a red fox."}) = 0.4 \times 0.27 \times 0.55 \times 0.79 = 0.0469$$

**Issue:** the longer the sentence, the lower its probability (a product of factors < 1).
**Fix:** normalize by the number of words using the **geometric mean**:
$$P_{norm}(W) = P(W)^{1/n} \quad\Rightarrow\quad P_{norm}(\text{"a red fox."}) = 0.0469^{1/4} = 0.465$$
$$\text{Perplexity}(W) = \frac{1}{P_{norm}(W)} = \frac{1}{P(W)^{1/n}} = \frac{1}{0.465} \approx 2.15$$

### General definition
Measures how well a probability model predicts a sample; the model's uncertainty in predicting the next token → **lower is better**. For held-out text $y_{1:M} = (y_1, \dots, y_M)$:
$$\text{Perplexity}(y_{1:M}) = 2^{-\frac{1}{M}L(y_{1:M})}, \qquad L(y_{1:M}) = \sum_{t=1}^{M}\log_2 p(y_t \mid y_{<t}) \;\;\text{(log-likelihood of the text)}$$
A better model has **higher log-likelihood** and **lower perplexity**. Perplexity = $2^{\text{cross entropy in bits}}$ (or $e^{\text{cross entropy in nats}}$, same value).

- **Best perplexity:** a perfect model gives probability 1 to every correct token → log-probability 0 → **perplexity = 1**.
- **Worst perplexity:** the model knows nothing and gives every token $\frac{1}{|V|}$:
$$\text{Perplexity} = 2^{-\frac{1}{M}\sum_{t=1}^{M}\log_2\frac{1}{|V|}} = 2^{-\log_2\frac{1}{|V|}} = |V|$$
Intuition: perplexity ≈ **how many words the model is choosing between** at each step.

In [12]:
vocab6 = ["a", "the", "red", "fox", "dog", "."]
step_probs = {                       # rows of the slide's tables
    ():                       [0.4, 0.3, 0.15, 0.08, 0.07, 0.0],
    ("a",):                   [0.01, 0.01, 0.27, 0.3, 0.4, 0.01],
    ("a", "red"):             [0.02, 0.03, 0.03, 0.55, 0.22, 0.15],
    ("a", "red", "fox"):      [0.02, 0.03, 0.03, 0.02, 0.11, 0.79],
}
sentence = ["a", "red", "fox", "."]
picked = [step_probs[tuple(sentence[:t])][vocab6.index(w)] for t, w in enumerate(sentence)]
P_W = np.prod(picked)
n = len(sentence)
print('picked probabilities  :', picked)
print(f'P(W)                  = {P_W:.4f}')
print(f'P_norm(W) = P(W)^(1/n) = {P_W ** (1 / n):.3f}')
print(f'Perplexity            = {1 / P_W ** (1 / n):.3f}')
L = np.sum(np.log2(picked))
print(f'via 2^(-L/M)          = {2 ** (-L / n):.3f}   (L = {L:.3f} bits)')
print(f'\nbest (perfect model)  = {2 ** (-np.log2(1.0)):.0f}')
print(f'worst (uniform, |V|=6) = {2 ** (-np.mean(np.log2([1 / 6] * n))):.0f}')

picked probabilities  : [0.4, 0.27, 0.55, 0.79]
P(W)                  = 0.0469
P_norm(W) = P(W)^(1/n) = 0.465
Perplexity            = 2.149
via 2^(-L/M)          = 2.149   (L = -4.413 bits)

best (perfect model)  = 1
worst (uniform, |V|=6) = 6


In [13]:
# Perplexity of our n-gram models on held-out sentences (smoothing is needed, or unseen n-grams give P = 0)
test = ["the dog sat on the mat .", "a cat sat on a rug .", "the fish is on the mat ."]

def perplexity(sentences, prob_fn):
    log2_sum, M = 0, 0
    for s in sentences:
        toks = ['<s>', '<s>'] + s.split() + ['</s>']
        for i in range(2, len(toks)):
            p = prob_fn(toks[i], toks[i - 2:i])
            log2_sum += math.log2(p); M += 1
    return 2 ** (-log2_sum / M)

models = {
    'unigram (add-1)':       lambda w, c: p_smooth(w, [], 1),
    'bigram (add-1)':        lambda w, c: p_smooth(w, c[-1:], 1),
    'trigram (add-1)':       lambda w, c: p_smooth(w, c, 1),
    'trigram (add-0.05)':    lambda w, c: p_smooth(w, c, 0.05),
    'interpolated (λ)':      lambda w, c: 0.1 / len(V) + 0.9 * p_interp(w, c, (0.2, 0.3, 0.5, 0.0)),   # + a bit of uniform so nothing is 0
    'uniform 1/|V|':         lambda w, c: 1 / len(V),
}
for name, fn in models.items():
    print(f'{name:22} perplexity = {perplexity(test, fn):6.2f}')
print(f'\n|V| = {len(V)}  -> the uniform model hits the "worst case" perplexity exactly')

unigram (add-1)        perplexity =  11.97
bigram (add-1)         perplexity =   5.70
trigram (add-1)        perplexity =   7.23
trigram (add-0.05)     perplexity =   3.43
interpolated (λ)       perplexity =   3.07
uniform 1/|V|          perplexity =  15.00

|V| = 15  -> the uniform model hits the "worst case" perplexity exactly


In [ ]:
# Perplexity of GPT-2 on a sentence (exp of the average cross-entropy loss)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained("gpt2"); gpt2 = AutoModelForCausalLM.from_pretrained("gpt2")
for s in ["There is a small nodule in the left lung.", "Lung the nodule small left a in there is."]:
    ids = tok(s, return_tensors="pt").input_ids
    with torch.no_grad():
        loss = gpt2(ids, labels=ids).loss          # mean cross entropy (nats)
    print(f'{s:45} perplexity = {torch.exp(loss).item():.1f}')

## BLEU score (Bilingual Evaluation Understudy)
**Definition:** compares the **n-grams** of a candidate translation with the n-grams of the **reference** text and counts matches (unigram = each token, bigram = each word pair).
- Proposed by **Kishore Papineni et al., 2002**, *"BLEU: a Method for Automatic Evaluation of Machine Translation"*.
- $0 \le \text{BLEU} \le 1$: closer to 1 is better. Reaching 1 is practically impossible; **> 0.3** is usually considered good.

### Precision
$$P = \frac{TP}{TP + FP}$$
- **TP**: n-grams in the candidate that match the reference · **FP**: candidate n-grams not in the reference

R: *The cat is on the mat.* · C: *The cat and the dog.* → matching unigrams: The, cat, the → $\text{BLEU}(C) = 3/5 = 0.6$

**Problem with repeated unigrams:** C: *The the the the the.* → every "the" is in the reference → $\text{BLEU}(C) = 5/5 = 1$ ❌

### Modified precision: clip the counts
$$\text{Count}_{clip} = \min(\text{count}, \text{Max\_Ref\_Count})$$
"the" appears at most **2** times in the reference → $\text{BLEU}(C) = 2/5 = 0.4$ ✅

### Considering n-grams
R: *There is a cat on the mat.* · C1: *There is a cat on the mat.* · C2: *Mat the cat is on a there.*
- Unigrams: $\text{BLEU}_1(C1) = 1$, $\text{BLEU}_1(C2) = 1$ ❌ (word order ignored)
- Bigrams: $\text{BLEU}_2(C1) = 1$, $\text{BLEU}_2(C2) = 0$ ✅

$$p_n = \frac{\sum_{C \in \{Candidates\}}\sum_{n\text{-gram} \in C}\text{Count}_{clip}(n\text{-gram})}{\sum_{C' \in \{Candidates\}}\sum_{n\text{-gram}' \in C'}\text{Count}(n\text{-gram}')}, \qquad n = 1..4$$
Small n-grams capture the sentence **meaning** (adequacy); large n-grams capture **well-formed** sentences (fluency).

### Brevity penalty (BP)
Short candidates can still have perfect precision: R: *There is a cat on the mat.* · C: *There is a cat.* → $\text{BLEU}_1 = 4/4 = 1$, $\text{BLEU}_2 = 3/3 = 1$ ❌

$$BP = \begin{cases} 1 & \text{if } c > r \\ e^{(1 - r/c)} & \text{if } c \le r \end{cases}$$
$c$: candidate length · $r$: effective reference length. If the candidate is shorter, $r/c > 1$, so $(1 - r/c) < 0$ and the penalty decays exponentially.

$$\text{BLEU} = BP \cdot \exp\Big(\sum_{n=1}^{N} w_n \log p_n\Big)$$
$N$: the n-gram order used (usually 4), $w_n$: weights (usually uniform, $1/N$).

In [14]:
def tokenize(s):
    return re.findall(r"[a-z]+", s.lower())

def modified_precision(cand, refs, n, clip=True):
    c_ngrams = Counter(ngrams(tokenize(cand), n))
    if not c_ngrams: return 0.0
    max_ref = Counter()
    for r in refs:
        for g, cnt in Counter(ngrams(tokenize(r), n)).items():
            max_ref[g] = max(max_ref[g], cnt)
    matched = sum(min(cnt, max_ref[g]) if clip else (cnt if g in max_ref else 0) for g, cnt in c_ngrams.items())
    return matched / sum(c_ngrams.values())

def brevity_penalty(cand, refs):
    c = len(tokenize(cand))
    r = min((len(tokenize(x)) for x in refs), key=lambda L: (abs(L - c), L))   # closest reference length
    return 1.0 if c > r else math.exp(1 - r / c)

def bleu(cand, refs, N=4):
    ps = [modified_precision(cand, refs, n) for n in range(1, N + 1)]
    if min(ps) == 0: return 0.0
    return brevity_penalty(cand, refs) * math.exp(sum(math.log(p) / N for p in ps))

R = ["The cat is on the mat."]
print('C = "The cat and the dog."  precision          :', modified_precision("The cat and the dog.", R, 1))
print('C = "The the the the the."  precision (no clip):', modified_precision("The the the the the.", R, 1, clip=False))
print('C = "The the the the the."  modified precision :', modified_precision("The the the the the.", R, 1))

R2 = ["There is a cat on the mat."]
for C in ["There is a cat on the mat.", "Mat the cat is on a there.", "There is a cat."]:
    p1, p2 = modified_precision(C, R2, 1), modified_precision(C, R2, 2)
    print(f'{C:28} BLEU1 = {p1:.2f}  BLEU2 = {p2:.2f}  BP = {brevity_penalty(C, R2):.3f}  BLEU-4 = {bleu(C, R2):.3f}')

C = "The cat and the dog."  precision          : 0.6
C = "The the the the the."  precision (no clip): 1.0
C = "The the the the the."  modified precision : 0.4
There is a cat on the mat.   BLEU1 = 1.00  BLEU2 = 1.00  BP = 1.000  BLEU-4 = 1.000
Mat the cat is on a there.   BLEU1 = 1.00  BLEU2 = 0.00  BP = 1.000  BLEU-4 = 0.000
There is a cat.              BLEU1 = 1.00  BLEU2 = 1.00  BP = 0.472  BLEU-4 = 0.472


In [ ]:
# Cross-check with NLTK (note: NLTK uses the same clipping + brevity penalty)
from nltk.translate.bleu_score import sentence_bleu
ref = [tokenize("There is a cat on the mat.")]
print(sentence_bleu(ref, tokenize("There is a cat on the mat.")))
print(sentence_bleu(ref, tokenize("There is a cat."), weights=(0.5, 0.5)))   # BLEU-2 with brevity penalty

# Implementation Strategies for Downstream Tasks
| Strategy | Idea |
|---|---|
| **Training from scratch** | use a large corpus to train a model from scratch |
| **Fine-tuning** | further train a **pre-trained** model on a smaller, task-specific dataset so it adapts to the task |
| **Feature extraction** | use the model's **embeddings** as input features for other ML models (e.g. logistic regression from L5) |
| **Prompt engineering** | design prompts that guide the model to the desired output using its pre-trained knowledge; especially popular with GPT-like models |

### Types of fine-tuning
| Type | What gets trained |
|---|---|
| **Full model fine-tuning** | the **entire** pre-trained model is trained further on the task data |
| **Partial fine-tuning** | only the **last few layers**; the rest are **frozen** |
| **Feature-based fine-tuning** | the output of one or more pre-trained layers feeds a **new model trained from scratch** |
| **Adapter modules** | small trainable layers **inserted between** pre-trained layers; only adapters are updated, the original model is frozen |
| **Task-specific heads** | add a new output layer ("head"); the pre-trained model **and** the head are updated together |

(LLM training pipelines also add instruction tuning and **RLHF** on top of pre-training.)

In [ ]:
# Sketch: feature extraction vs fine-tuning with BERT (needs transformers + torch)
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from sklearn.linear_model import LogisticRegression

texts = ["no acute findings", "large mass in the left lung", "normal study", "suspicious nodule seen"]
labels = [0, 1, 0, 1]
tok = AutoTokenizer.from_pretrained("bert-base-uncased")

# 1) Feature extraction: frozen BERT [CLS] vectors -> logistic regression
bert = AutoModel.from_pretrained("bert-base-uncased").eval()
with torch.no_grad():
    feats = bert(**tok(texts, padding=True, return_tensors="pt")).last_hidden_state[:, 0]   # [CLS]
clf = LogisticRegression(max_iter=1000).fit(feats.numpy(), labels)

# 2) Task-specific head + fine-tuning (partial: freeze everything except the last layer and the head)
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
for name, p in model.named_parameters():
    p.requires_grad = name.startswith("classifier") or "encoder.layer.11" in name
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-5)
batch = tok(texts, padding=True, return_tensors="pt")
for epoch in range(3):
    out = model(**batch, labels=torch.tensor(labels))
    out.loss.backward(); opt.step(); opt.zero_grad()
    print(epoch, out.loss.item())

# Summary
| | n-gram | FFNN | RNN | CNN | Transformer |
|---|---|---|---|---|---|
| Context | last $n-1$ words | fixed window | whole prefix (in theory) | stacked local windows | whole sequence (attention) |
| Probabilities | counted | predicted | predicted | predicted | predicted |
| Parallel training | — | yes | no (sequential) | yes | yes |
| Main issue | sparsity, storage $O(\exp(n))$ | fixed window | slow, vanishing gradients | limited receptive field | cost grows with length² |

**Evaluation:** cross entropy (lower is better) · perplexity = 2^cross-entropy (1 = perfect, |V| = clueless) · BLEU (n-gram overlap with clipping + brevity penalty).

# References
- Lena Voita, [NLP Course: Language Modeling](https://lena-voita.github.io/nlp_course/language_modeling.html)
- Jay Alammar, [The Illustrated GPT-2](https://jalammar.github.io/illustrated-gpt2/)
- Bengio et al., 2003, *A Neural Probabilistic Language Model*
- Devlin et al., 2018, *BERT*; Radford et al., 2018, *Improving Language Understanding by Generative Pre-Training*; Raffel et al., 2019, *T5*
- Papineni et al., 2002, *BLEU: a Method for Automatic Evaluation of Machine Translation*
- "Understanding Convolutional Neural Networks for Text Classification"
- Survey: *Examining User-Friendly and Open-Sourced Large GPT Models* ([arXiv 2308.14149](https://arxiv.org/pdf/2308.14149.pdf))